
# Colab quickstart + RunManager for the TRPO / PPO repo

This notebook keeps the original Colab quickstart flow, but adds a **RunManager layer** so you can:

- launch one run with one function call
- save checkpoints locally on Colab SSD while training
- periodically sync outputs to Google Drive **during** training
- do a final sync automatically when training ends
- create `best.pt` and `last.pt` aliases after the run
- batch over locomotion seeds or Atari games with small loops

The intended workflow is:
1. mount Drive
2. copy the repo from Drive to `/content/trpo`
3. install dependencies
4. define the run manager helpers
5. launch runs from one function call


## 1. Mount Google Drive

In [ ]:

from google.colab import drive

drive.mount('/content/drive')


## 2. Copy the repo from Drive to Colab local SSD

In [ ]:

# Adjust these paths if your Drive layout is different.
DRIVE_REPO_PATH = "/content/drive/MyDrive/Colab Notebooks/839/trpo"
LOCAL_REPO_PATH = "/content/trpo"

!rm -rf "$LOCAL_REPO_PATH"
!cp -r "$DRIVE_REPO_PATH" "$LOCAL_REPO_PATH"
%cd "$LOCAL_REPO_PATH"


## 3. Install dependencies inside the Colab runtime

In [ ]:

!python3 -m pip install -r requirements.txt


## 4. Check CUDA and available CPU resources

In [ ]:

import os
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('CPU count:', os.cpu_count())



## 5. Configure default paths

These are the two output roots you will care about most:

- `LOCAL_RUNS_ROOT`: fast Colab SSD location used during training
- `DRIVE_RUNS_ROOT`: Google Drive location where finished and in-progress runs get synced


In [ ]:

from pathlib import Path

LOCAL_RUNS_ROOT = Path('/content/trpo_runs')
DRIVE_RUNS_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/839/trpo_outputs')

LOCAL_RUNS_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_RUNS_ROOT.mkdir(parents=True, exist_ok=True)

print('Local runs root :', LOCAL_RUNS_ROOT)
print('Drive runs root :', DRIVE_RUNS_ROOT)


## 6. RunManager helpers

In [ ]:

import csv
import json
import os
import shutil
import signal
import subprocess
import sys
import threading
import time
from pathlib import Path
from typing import Any, Iterable

REPO_ROOT = Path('/content/trpo')
TRAIN_SCRIPT = REPO_ROOT / 'scripts' / 'train.py'
EVAL_SCRIPT = REPO_ROOT / 'scripts' / 'evaluate.py'
AGGREGATE_SCRIPT = REPO_ROOT / 'scripts' / 'aggregate_results.py'


def _canonicalize_value(value: Any) -> str:
    if isinstance(value, bool):
        return 'true' if value else 'false'
    return str(value)


def print_shell_command(cmd: list[str]) -> None:
    pretty = ' \\
  '.join(cmd)
    print(pretty)


def make_run_name(config_path: str, seed: int, tag: str | None = None) -> str:
    stem = Path(config_path).stem
    return f"{stem}{'_' + tag if tag else ''}/seed_{seed}"


def sync_dir(src: Path, dst: Path, delete: bool = False) -> None:
    src = Path(src)
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)

    rsync = shutil.which('rsync')
    if rsync is not None:
        cmd = [rsync, '-a']
        if delete:
            cmd.append('--delete')
        cmd += [f'{src}/', f'{dst}/']
        subprocess.run(cmd, check=True)
        return

    if delete and dst.exists():
        shutil.rmtree(dst)
    dst.mkdir(parents=True, exist_ok=True)

    for root, dirs, files in os.walk(src):
        rel = Path(root).relative_to(src)
        (dst / rel).mkdir(parents=True, exist_ok=True)
        for file_name in files:
            shutil.copy2(Path(root) / file_name, dst / rel / file_name)


def periodic_sync_worker(local_dir: Path, drive_dir: Path, stop_event: threading.Event, every_seconds: int = 300):
    while not stop_event.wait(every_seconds):
        try:
            sync_dir(local_dir, drive_dir, delete=False)
            print(f'[sync] periodic sync complete -> {drive_dir}')
        except Exception as exc:
            print(f'[sync] periodic sync failed: {exc}')


def read_metrics_rows(run_dir: Path) -> list[dict[str, str]]:
    csv_path = run_dir / 'metrics.csv'
    if not csv_path.exists():
        return []
    with csv_path.open('r', encoding='utf-8', newline='') as f:
        return list(csv.DictReader(f))


def available_checkpoints(run_dir: Path) -> list[Path]:
    ckpt_dir = run_dir / 'checkpoints'
    if not ckpt_dir.exists():
        return []
    return sorted(ckpt_dir.glob('epoch_*.pt'))


def find_last_checkpoint(run_dir: Path) -> Path | None:
    ckpts = available_checkpoints(run_dir)
    return ckpts[-1] if ckpts else None


def find_best_checkpoint(run_dir: Path, metric: str = 'train_return_mean', maximize: bool = True) -> tuple[Path | None, dict[str, Any] | None]:
    rows = read_metrics_rows(run_dir)
    if not rows:
        return None, None

    existing = {p.stem.replace('epoch_', ''): p for p in available_checkpoints(run_dir)}
    best_row = None
    best_score = None
    best_ckpt = None

    for row in rows:
        epoch = row.get('epoch') or row.get('iteration')
        if epoch is None:
            continue
        epoch_key = str(epoch).zfill(4)
        ckpt = existing.get(epoch_key)
        if ckpt is None:
            continue
        value = row.get(metric)
        if value in (None, '', 'nan', 'NaN'):
            continue
        try:
            score = float(value)
        except ValueError:
            continue
        if best_score is None or (score > best_score if maximize else score < best_score):
            best_score = score
            best_row = row
            best_ckpt = ckpt

    return best_ckpt, best_row


def write_checkpoint_aliases(run_dir: Path, metric: str = 'train_return_mean', maximize: bool = True) -> dict[str, str | None]:
    ckpt_dir = run_dir / 'checkpoints'
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    last_ckpt = find_last_checkpoint(run_dir)
    best_ckpt, best_row = find_best_checkpoint(run_dir, metric=metric, maximize=maximize)

    aliases = {'last_checkpoint': None, 'best_checkpoint': None}

    if last_ckpt is not None:
        dst = ckpt_dir / 'last.pt'
        shutil.copy2(last_ckpt, dst)
        aliases['last_checkpoint'] = str(dst)

    if best_ckpt is not None:
        dst = ckpt_dir / 'best.pt'
        shutil.copy2(best_ckpt, dst)
        aliases['best_checkpoint'] = str(dst)
        with (ckpt_dir / 'best_checkpoint_info.json').open('w', encoding='utf-8') as f:
            json.dump({'checkpoint': str(best_ckpt), 'metric_row': best_row}, f, indent=2)

    return aliases


def save_run_summary(run_dir: Path, drive_dir: Path, cmd: list[str], extra: dict[str, Any] | None = None) -> Path:
    summary_path = run_dir / 'run_manager_summary.json'
    payload = {
        'run_dir': str(run_dir),
        'drive_dir': str(drive_dir),
        'command': cmd,
        'timestamp_unix': time.time(),
    }
    if extra:
        payload.update(extra)
    with summary_path.open('w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2)
    return summary_path


def run_training(
    config_path: str,
    seed: int = 0,
    output_tag: str | None = None,
    local_runs_root: Path | str = LOCAL_RUNS_ROOT,
    drive_runs_root: Path | str = DRIVE_RUNS_ROOT,
    sync_every_seconds: int = 300,
    periodic_sync: bool = True,
    best_metric: str = 'train_return_mean',
    maximize_metric: bool = True,
    extra_args: dict[str, Any] | None = None,
) -> dict[str, Any]:
    local_runs_root = Path(local_runs_root)
    drive_runs_root = Path(drive_runs_root)

    run_name = make_run_name(config_path=config_path, seed=seed, tag=output_tag)
    local_run_dir = local_runs_root / run_name
    drive_run_dir = drive_runs_root / run_name
    local_run_dir.parent.mkdir(parents=True, exist_ok=True)
    drive_run_dir.parent.mkdir(parents=True, exist_ok=True)

    cmd = [
        sys.executable,
        str(TRAIN_SCRIPT),
        '--config', config_path,
        '--seed', str(seed),
        '--output-dir', str(local_run_dir),
        '--overwrite',
    ]

    if extra_args:
        for key, value in extra_args.items():
            key = str(key)
            if not key.startswith('--'):
                raise ValueError(f'Expected CLI key starting with --, got: {key}')
            if value is None:
                continue
            if isinstance(value, bool):
                if value:
                    cmd.append(key)
            else:
                cmd.extend([key, _canonicalize_value(value)])

    print('Launching run:')
    print_shell_command(cmd)
    print('Local run dir :', local_run_dir)
    print('Drive run dir :', drive_run_dir)

    stop_event = threading.Event()
    sync_thread = None
    if periodic_sync:
        sync_thread = threading.Thread(
            target=periodic_sync_worker,
            args=(local_run_dir, drive_run_dir, stop_event, sync_every_seconds),
            daemon=True,
        )
        sync_thread.start()

    start = time.time()
    proc = subprocess.Popen(cmd, cwd=str(REPO_ROOT))
    return_code = None

    try:
        return_code = proc.wait()
    except KeyboardInterrupt:
        print('\n[run] Interrupted. Terminating training process...')
        proc.send_signal(signal.SIGINT)
        try:
            return_code = proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            proc.kill()
            return_code = proc.wait()
        raise
    finally:
        stop_event.set()
        if sync_thread is not None:
            sync_thread.join(timeout=5)

    elapsed = time.time() - start
    if return_code != 0:
        print(f'[run] Training exited with code {return_code}. Performing final sync anyway...')

    aliases = write_checkpoint_aliases(local_run_dir, metric=best_metric, maximize=maximize_metric)
    summary_path = save_run_summary(
        local_run_dir,
        drive_run_dir,
        cmd,
        extra={
            'return_code': return_code,
            'elapsed_sec': elapsed,
            'aliases': aliases,
        },
    )

    sync_dir(local_run_dir, drive_run_dir, delete=False)
    print(f'[sync] final sync complete -> {drive_run_dir}')

    result = {
        'return_code': return_code,
        'elapsed_sec': elapsed,
        'local_run_dir': str(local_run_dir),
        'drive_run_dir': str(drive_run_dir),
        'summary_path': str(summary_path),
        **aliases,
    }
    print(json.dumps(result, indent=2))
    return result


def aggregate_runs(
    runs_roots: Iterable[str | Path],
    compare: bool = False,
    metric: str = 'train_return_mean',
    x_axis: str = 'iteration',
    smooth_window: int | None = None,
    summary: bool = True,
):
    cmd = [sys.executable, str(AGGREGATE_SCRIPT)]
    for root in runs_roots:
        cmd.extend(['--runs-root', str(root)])
    if compare:
        cmd.append('--compare')
    cmd.extend(['--metric', metric, '--x-axis', x_axis])
    if smooth_window is not None:
        cmd.extend(['--smooth-window', str(smooth_window)])
    if summary:
        cmd.append('--summary')
    print_shell_command(cmd)
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True)


def evaluate_checkpoint(
    config_path: str,
    checkpoint_path: str | Path,
    episodes: int = 5,
    device: str = 'cuda',
    extra_args: dict[str, Any] | None = None,
):
    cmd = [
        sys.executable,
        str(EVAL_SCRIPT),
        '--config', config_path,
        '--checkpoint', str(checkpoint_path),
        '--episodes', str(episodes),
        '--device', device,
    ]
    if extra_args:
        for key, value in extra_args.items():
            if value is None:
                continue
            cmd.extend([str(key), _canonicalize_value(value)])
    print_shell_command(cmd)
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True)



## 7. Recommended argument presets

These helpers just keep the command lines short. Adjust them if you want.


In [ ]:

ATARI_TRPO_SAFE_ARGS = {
    '--num-workers': 4,
    '--memory-mode': 'safe',
    '--obs-storage': 'ram',
    '--full-batch-chunk-size': 8192,
    '--device': 'cuda',
    '--progress-mode': 'off',
}

ATARI_PPO_ARGS = {
    '--num-workers': 4,
    '--memory-mode': 'standard',
    '--device': 'cuda',
    '--progress-mode': 'off',
}

MUJOCO_TRPO_ARGS = {
    '--num-workers': 12,
    '--memory-mode': 'standard',
    '--device': 'cuda',
    '--progress-mode': 'off',
}

MUJOCO_NPG_ARGS = {
    '--num-workers': 12,
    '--memory-mode': 'standard',
    '--device': 'cuda',
    '--progress-mode': 'off',
}

MUJOCO_PPO_ARGS = {
    '--num-workers': 12,
    '--memory-mode': 'standard',
    '--device': 'cuda',
    '--progress-mode': 'off',
}



## 8. Single-run examples

The point is that each run should be one function call.


In [ ]:

# Example: Atari TRPO single-path
# result = run_training(
#     config_path='configs/atari/seaquest_single_path.yaml',
#     seed=0,
#     extra_args={
#         **ATARI_TRPO_SAFE_ARGS,
#         '--epochs': 300,
#         '--save-interval': 25,
#     },
# )
# result


In [ ]:

# Example: Atari PPO clip
# result = run_training(
#     config_path='configs/atari/seaquest_ppo_clip.yaml',
#     seed=0,
#     extra_args={
#         **ATARI_PPO_ARGS,
#         '--epochs': 300,
#         '--save-interval': 25,
#     },
# )
# result


In [ ]:

# Example: MuJoCo TRPO single-path
# result = run_training(
#     config_path='configs/mujoco/walker2d_single_path.yaml',
#     seed=0,
#     extra_args={
#         **MUJOCO_TRPO_ARGS,
#         '--epochs': 200,
#         '--save-interval': 25,
#     },
# )
# result


## 9. Batch helpers for locomotion and Atari

In [ ]:

# Locomotion batch plan: 3 seeds each for the three main locomotion tasks.
LOCOMOTION_BATCH = [
    ('configs/mujoco/swimmer_single_path.yaml', [0, 1, 2], MUJOCO_TRPO_ARGS, {'--epochs': 200, '--save-interval': 25}),
    ('configs/mujoco/hopper_single_path.yaml', [0, 1, 2], MUJOCO_TRPO_ARGS, {'--epochs': 200, '--save-interval': 25}),
    ('configs/mujoco/walker2d_single_path.yaml', [0, 1, 2], MUJOCO_TRPO_ARGS, {'--epochs': 200, '--save-interval': 25}),
]

# Atari batch plan: 1 seed each because runs are much more expensive.
ATARI_BATCH = [
    ('configs/atari/beamrider_single_path.yaml', [0], ATARI_TRPO_SAFE_ARGS, {'--epochs': 300, '--save-interval': 25}),
    ('configs/atari/breakout_single_path.yaml', [0], ATARI_TRPO_SAFE_ARGS, {'--epochs': 300, '--save-interval': 25}),
    ('configs/atari/enduro_single_path.yaml', [0], ATARI_TRPO_SAFE_ARGS, {'--epochs': 300, '--save-interval': 25}),
    ('configs/atari/pong_single_path.yaml', [0], ATARI_TRPO_SAFE_ARGS, {'--epochs': 300, '--save-interval': 25}),
    ('configs/atari/qbert_single_path.yaml', [0], ATARI_TRPO_SAFE_ARGS, {'--epochs': 300, '--save-interval': 25}),
    ('configs/atari/seaquest_single_path.yaml', [0], ATARI_TRPO_SAFE_ARGS, {'--epochs': 300, '--save-interval': 25}),
    ('configs/atari/spaceinvaders_single_path.yaml', [0], ATARI_TRPO_SAFE_ARGS, {'--epochs': 300, '--save-interval': 25}),
]


In [ ]:

def run_batch(batch_spec, output_tag: str | None = None):
    results = []
    for config_path, seeds, preset_args, extra in batch_spec:
        for seed in seeds:
            print('
' + '=' * 100)
            print(f'Running: config={config_path} seed={seed}')
            res = run_training(
                config_path=config_path,
                seed=seed,
                output_tag=output_tag,
                extra_args={**preset_args, **extra},
            )
            results.append(res)
    return results


In [ ]:

# Uncomment one of these when you are ready.

# locomotion_results = run_batch(LOCOMOTION_BATCH)
# atari_results = run_batch(ATARI_BATCH)


## 10. Aggregation / evaluation helpers

In [ ]:

# Example aggregation for locomotion comparison.
# aggregate_runs(
#     runs_roots=[
#         LOCAL_RUNS_ROOT / 'swimmer_single_path',
#         LOCAL_RUNS_ROOT / 'swimmer_natural_pg',
#         LOCAL_RUNS_ROOT / 'swimmer_ppo_clip',
#     ],
#     compare=True,
#     metric='train_return_mean',
#     x_axis='iteration',
#     smooth_window=5,
#     summary=True,
# )


In [ ]:

# Example evaluation of the best checkpoint from a finished run.
# run_dir = LOCAL_RUNS_ROOT / 'seaquest_single_path' / 'seed_0'
# best_ckpt = run_dir / 'checkpoints' / 'best.pt'
# evaluate_checkpoint(
#     config_path='configs/atari/seaquest_single_path.yaml',
#     checkpoint_path=best_ckpt,
#     episodes=5,
#     device='cuda',
# )



## 11. Notes / practical advice

- The **periodic sync** is there to mitigate the risk of losing checkpoints if the runtime dies or disconnects.
- The **final sync** always runs after the training subprocess exits, even if the return code is non-zero.
- `best.pt` is chosen from epochs that actually have a saved checkpoint file.
- `last.pt` is the latest saved checkpoint file.
- For Atari TRPO on Colab, `memory_mode=safe` + `obs_storage=ram` + chunking is still the intended setup.
- For PPO, standard memory mode is usually fine.
- If a long run is precious, increase `--save-interval` only as much as you are comfortable with.
